# Recommender Systems - Part A Project
**Paper:** SVD-GoRank: Recommender System Algorithm Using SVD and Gower's Ranking

**Dataset:** Book-Crossing (BX)

**Team Members:** Harun Korkmaz, Muhammet Salih Hasılcıo, Orhan Efe Bayrak, Muhiddin Fırat, Yasin Furkan Abasız, Yakup Berkay Genceroğlu, Görkem Yahya Bakan

This notebook contains data preprocessing and matrix construction steps for the Book-Crossing dataset where users rate books on a scale of 0 to 10 (0 represents implicit/implicit ratings). This dataset is critical for demonstrating the power of SVD with its enormous sparsity rate of 99.99%.

## PROGRESS 1: Data Loading & Preprocessing
In this step, the original Book-Crossing dataset is loaded into the system using Pandas, unnecessary columns (timestamp) are removed, and statistical properties mentioned in the paper (especially Sparsity rate) are calculated.

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ruchi798/bookcrossing-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Harun\.cache\kagglehub\datasets\ruchi798\bookcrossing-dataset\versions\3


In [ ]:
import pandas as pd
import numpy as np

# 1. Load the Book-Crossing dataset
# Assume the file is named 'BX-Book-Ratings.csv' downloaded from Kaggle
file_path = r"C:\Users\Harun\.cache\kagglehub\datasets\ruchi798\bookcrossing-dataset\versions\3\Book reviews\Book reviews\BX-Book-Ratings.csv"

# NOTE: The delimiter is semicolon (;) and some special characters (letters in ISBN)
# Use encoding='latin-1' (or 'cp1252') to avoid errors
# Also set quotechar='"' because column names may contain quotes
df_bx = pd.read_csv(file_path, sep=';', encoding='latin-1', on_bad_lines='skip')

# Standardize column names with other datasets to avoid confusion
# when writing SVD and Gower algorithms later
df_bx.rename(columns={'User-ID': 'user_id', 'ISBN': 'item_id', 'Book-Rating': 'rating'}, inplace=True)

# Calculate statistics
n_users = df_bx['user_id'].nunique()
n_items = df_bx['item_id'].nunique()
n_ratings = len(df_bx)

# Sparsity Formula
sparsity = 1.0 - (n_ratings / (n_users * n_items))

print("--- BOOK-CROSSING - PROGRESS 1 ---")
print(f"Total Users: {n_users}")
print(f"Total Books (ISBN): {n_items}")
print(f"Total Ratings: {n_ratings}")
print(f"Sparsity Ratio: {sparsity:.6f} ({sparsity*100:.4f}%)")
print("\nFirst 5 Rows of Dataset (0 ratings represent implicit/hidden ratings):")
display(df_bx.head())

--- BOOK-CROSSING - PROGRESS 1 ---
Total Users: 105283
Total Books (ISBN): 340556
Total Ratings: 1149780
Sparsity Ratio: 0.999968 (99.9968%)

First 5 Rows of Dataset (0 ratings represent implicit/hidden ratings):


,user_id,item_id,rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


## PROGRESS 2: Train/Test Split & User-Item Matrix Creation
To fairly evaluate the model, we split the data into 80% training and 20% test sets. Then, we construct the "User-Item Interaction Matrix" which will be used in Phase-1 (SVD) of the paper and is currently mostly empty (NaN).

In [7]:
from sklearn.model_selection import train_test_split

# 1. Split data into 80% Training and 20% Testing
train_data_bx, test_data_bx = train_test_split(df_bx, test_size=0.20, random_state=42)

print("--- BOOK-CROSSING - PROGRESS 2 ---")
print(f"Training Set Size: {len(train_data_bx)} rows")
print(f"Test Set Size: {len(test_data_bx)} rows\n")

print("CRITICAL ENGINEERING NOTE")
print("Book-Crossing matrix dimensions are ~278,000 Users x ~271,000 Books.")
print("This means 75 billion cells, and Pandas .pivot() function will immediately cause")
print("out-of-memory (MemoryError) on standard computers.")
print("Therefore, matrix construction is deferred to the SVD phase using scipy.sparse library.\n")

# To show the professor, we demonstrate pivot operation on a small subset:
# First 1000 most-rated users and first 1000 most-rated books:

top_users = train_data_bx['user_id'].value_counts().index[:1000]
top_books = train_data_bx['item_id'].value_counts().index[:1000]

sample_df = train_data_bx[train_data_bx['user_id'].isin(top_users) & train_data_bx['item_id'].isin(top_books)]
sample_user_item_matrix = sample_df.pivot(index='user_id', columns='item_id', values='rating')

print(f"Sample (Reduced) User-Item Matrix Size: {sample_user_item_matrix.shape}")
print("Sample Matrix Visualization (Empty cells are NaN):")
display(sample_user_item_matrix.head())

--- BOOK-CROSSING - PROGRESS 2 ---
Training Set Size: 919824 rows
Test Set Size: 229956 rows

CRITICAL ENGINEERING NOTE
Book-Crossing matrix dimensions are ~278,000 Users x ~271,000 Books.
This means 75 billion cells, and Pandas .pivot() function will immediately cause
out-of-memory (MemoryError) on standard computers.
Therefore, matrix construction is deferred to the SVD phase using scipy.sparse library.

Sample (Reduced) User-Item Matrix Size: (980, 1000)
Sample Matrix Visualization (Empty cells are NaN):


item_id,000649840X,002542730X,0060008032,0060096195,006016848X,0060173289,0060175400,0060188731,006019491X,0060199652,...,1573225517,1573225789,1573227331,1573229326,1573229571,1573229725,1576737330,1592400876,1878424319,8873122933
user_id,,,,,,,,,,,,,,,,,,,,,
254,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2276,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2766,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2977,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
